In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Standardisert treningsoppsett for endometriekreft-tumorsegmentering
– støtter 1-3 modaliteter (VIBE, T2, ADC)
"""

# ==============================================================
# Importer biblioteker og parametere
# ==============================================================
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import params
from datasetgenerator import *   # din funksjon ovenfor
from torch.utils.data import DataLoader
import torchio as tio
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from fastai.vision.all import Learner, DataLoaders
from fastai.callback.tracker import EarlyStoppingCallback
from fastai.callback.core import Callback
from fastai.learner import Metric
from utils import *


2025-11-27 09:27:57.276799: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-27 09:27:57.340520: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-27 09:27:58.260609: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Hvilke modaliteter som skal brukes i dette eksperimentet
#selected_modalities = ["T2", "ADC"]
#selected_modalities = ["vibe2min", "T2", "ADC"]
selected_modalities = ["vibe2min"]

# Hvilket bilde som brukes som referanse
#reference = "T2"
reference = "vibe2min"


# ==============================================================
# 1. Les inn og filtrer datasett
# ==============================================================
df = pd.read_csv(params.pathlistvalid, sep=';').groupby("subj", as_index=False).first()

# Which columsn to require
require = [params.modalities[m]["col"] for m in selected_modalities]
df = datasetgenerator(df, require)

# Only data sets with manual masks
df = df.loc[df.dataset == 'man'].reset_index(drop=True)
print(f"Antall datasett med manuelle masker: {len(df)}")

# ==============================================================
# 2. Bygg dynamiske paths for bilder og masker
# ==============================================================

df["imgpath"] = [
    build_image_path(
        s,
        selected_modalities,
        params.prepathnifti,
        reference,
        params.modalities
    )
    for s in df.subj
]

df["pathmask"] = [
    build_mask_path(s, m, params.prepathnifti, reference)
    for s, m in zip(df.subj, df.pathmask)
]
df.head(3)

587 datasett tilfredsstiller betingelsene
Antall datasett med manuelle masker: 273


,subj,pathvibe2minDicom,pathT2Dicom,pathADCDicom,pathJADNifti,pathKWLNifti,pathVerifiedMLNifti,pathJADMLNifti,pathmask,dataset,imgpath
0,11,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,011segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC011/registered/011segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC011/unregistered/vibe2min.nii.gz
1,17,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,017segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC017/registered/017segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC017/unregistered/vibe2min.nii.gz
2,20,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,020segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC020/registered/020segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC020/unregistered/vibe2min.nii.gz


In [3]:
from tqdm import tqdm
print("🔍 Beregner tumorvolum for alle masker ...")
df["tumorsize"] = [compute_tumor_volume(p) for p in tqdm(df["pathmask"].values)]
print(f"✅ Volum beregnet for {df['tumorsize'].notna().sum()} pasienter.")


🔍 Beregner tumorvolum for alle masker ...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 273/273 [00:15<00:00, 17.13it/s]

✅ Volum beregnet for 273 pasienter.


In [4]:
# ==============================================================
# 3. Split datasett i train / val / test
# ==============================================================
# Del datasettet i trenings-, validerings- og test-sett.
# - test_fraction definerer hvor stor andel som holdes av til test.
# - validation_fraction brukes til å trekke ut et subset av trening til validering.

# Lag kategorier basert på tumorstørrelse (kvartiler)
df["tumorsize_cat"] = pd.qcut(
    df["tumorsize"],
    q=4,  # antall grupper (4=kvartiler)
    labels=["Q1_Smallest", "Q2", "Q3", "Q4_Largest"]
)

# Fjern NaN (masker som ikke kunne leses)
df = df.dropna(subset=["tumorsize_cat"]).reset_index(drop=True)
from sklearn.model_selection import train_test_split

test_fraction = 0.2
random_state = 42
validation_fraction = 0.1

dftrain, dftest = train_test_split(
    df,
    test_size=test_fraction,
    stratify=df["tumorsize_cat"],
    random_state=random_state
)

# Lag valideringssplit som før
val_idx = dftrain.sample(
    frac=validation_fraction,
    random_state=random_state
).index
dftrain["isval"] = False
dftrain.loc[val_idx, "isval"] = True

# Skriv ut fordeling
print(f"\n🧠 Modaliteter: {params.modalities}")
print(f"Train: {len(dftrain)},  Val: {dftrain['isval'].sum()},  Test: {len(dftest)}")

# Fordeling av tumorvolum per gruppe
print("\n📊 Tumorvolum-fordeling (ml):")
print(df.groupby("tumorsize_cat")["tumorsize"].describe()[["min", "max", "mean"]])



🧠 Modaliteter: {'vibe2min': {'col': 'pathvibe2minDicom', 'file': 'vibe2min.nii.gz'}, 'T2': {'col': 'pathT2Dicom', 'file': 'T2.nii.gz'}, 'ADC': {'col': 'pathADCDicom', 'file': 'ADC.nii.gz'}}
Train: 218,  Val: 22,  Test: 55

📊 Tumorvolum-fordeling (ml):
                     min         max       mean
tumorsize_cat                                  
Q1_Smallest     0.078970    3.340787   1.577923
Q2              3.362532    7.872995   5.636550
Q3              7.949676   19.056533  12.337547
Q4_Largest     19.280329  679.400909  62.928205


In [14]:
# ==========================================================
# Imports
# ==========================================================
import torch
import numpy as np
import torchio as tio
from torchio.data import SubjectsLoader
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fastai.callback.tracker import EarlyStoppingCallback
from fastai.callback.core import Callback

# ==========================================================
# Augmentations
# ==========================================================
def safe_multichannel_aug(img_size):
    return tio.Compose([
        tio.Resample(1),
        tio.ZNormalization(),
        tio.CropOrPad((img_size, img_size, img_size)),
        tio.RandomFlip(axes=(0,1,2), p=0.5),
        tio.RandomAffine(scales=(0.9,1.1), translation=2, degrees=5, p=0.25),
        tio.RandomElasticDeformation(num_control_points=5, max_displacement=1, p=0.15),
        tio.Lambda(lambda x: x + torch.randn_like(x) * 0.01, p=0.15),
    ])


# ==========================================================
# TorchIO dataloader
# ==========================================================
def get_dataloader(df, batch_size, img_size, augment=False):
    transforms = safe_multichannel_aug(img_size) if augment else tio.Compose([
        tio.Resample(1),
        tio.ZNormalization(),
        tio.CropOrPad((img_size, img_size, img_size)),
    ])

    subjects = []
    for imgs, mask in zip(df.imgpath.values, df.pathmask.values):
        img_files = imgs.split(";")
        subjects.append(
            tio.Subject(
                images=tio.ScalarImage(img_files),
                mask=tio.LabelMap(mask),
            )
        )

    dataset = tio.SubjectsDataset(subjects, transform=transforms)

    return SubjectsLoader(
        dataset,
        batch_size=batch_size,
        shuffle=augment,
        num_workers=4,
        pin_memory=True,
    )


# ==========================================================
# Callback som flytter Subject batch til valgt GPU
# ==========================================================
class SubjectToDevice(Callback):
    order = 5
    def before_batch(self):
        device = self.learn.device

        # TorchIO gir dict { 'images': Tensor, 'mask': Tensor }
        xb = self.xb[0]   # fastai gir tuple
        yb = self.yb[0]

        self.xb = (xb.to(device),)
        self.yb = (yb.to(device),)


# ==========================================================
# PARAMETRE
# ==========================================================
GPU_ID         = 1
batch_size     = 4
img_size       = 144
n_epochs       = 200
learning_rate  = 1e-3
monitor_metric = 'dice'
patience       = 10


# ==========================================================
# VELG GPU
# ==========================================================
assert torch.cuda.is_available(), "CUDA er ikke tilgjengelig!"
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")

print(f"→ Bruker GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")


# ==========================================================
# Loaders
# ==========================================================
train_loader = get_dataloader(
    dftrain[dftrain.isval == False],
    batch_size,
    img_size,
    augment=True,
)

val_loader = get_dataloader(
    dftrain[dftrain.isval == True],
    batch_size,
    img_size,
    augment=False,
)

dls = DataLoaders(train_loader, val_loader, device=None)   # device=None = vi håndterer det selv
dls.n_inp = 1


# ==========================================================
# Modell
# ==========================================================
model = monai_unet_model(
    in_channels=len(selected_modalities)
).to(device)


# ==========================================================
# Learner
# ==========================================================
learn = Learner(
    dls,
    model,
    loss_func=DiceLoss(sigmoid=True),
    metrics=[ThresholdedDice()],
    cbs=[
        SubjectToDevice(),  # ← INGEN arguments!
        EarlyStoppingCallback(
            monitor=monitor_metric,
            comp=np.greater,
            patience=patience
        ),
    ]
)

learn.to(device)


# ==========================================================
# Tren!
# ==========================================================
learn.fit_one_cycle(n_epochs, lr_max=learning_rate)


→ Bruker GPU 1: Tesla V100-DGXS-32GB


epoch,train_loss,valid_loss,dice,time


Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyTorch DataLoader with a torchio.SubjectsLoader so that the collated batch becomes a dictionary, as expected. See https://github.com/fepegar/torchio/issues/1179 for more context about this issue.
Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyTorch DataLoader with a torchio.SubjectsLoader so that the collated batch becomes a dictionary, as expected. See https://github.com/fepegar/torchio/issues/1179 for more context about this issue.
Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyT

TypeError: unhashable type: 'slice'

In [ ]:
import matplotlib.pyplot as plt

# Plot loss-kurven
learn.recorder.plot_loss()

# Plot Dice-score over epoker (hvis du har metrikken registrert)
metrics = learn.recorder.values
epochs = range(1, len(metrics)+1)
train_loss = [m[0] for m in metrics]
val_loss   = [m[1] for m in metrics]
dice       = [m[2] for m in metrics]  # Assuming [train_loss, valid_loss, dice]

plt.figure(figsize=(8,5))
plt.plot(epochs, dice, 'o-', label='Validation Dice')
plt.xlabel('Epoch')
plt.ylabel('Dice score')
plt.title('Validation Dice per epoch')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Save the model
version = "v0.1"
basename = os.path.join(
    params.prepathmodels,
    f"{params.modelname}_Monai_{'_'.join(params.selected_modalities)}_{version}"
)
model_path = basename + '.pth'

torch.save(model.state_dict(), model_path)
print(f"\n✅ Modell lagret til: {model_path}")


In [ ]:
# ==============================================================
# 8. Lagring av modellinnstillinger og datasett
# ==============================================================

import datetime

# Hent parametere
bs = params.batch_size
img_size = params.img_size
dropout = params.dropout_rate
lr = params.learning_rate
n_epoch = 50
#modelname = "MONAI_3D_UNet"
gt = np.greater

# Basename for filnavn (unik identifikator)
#timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
#basename = f"unet_{'_'.join(params.selected_modalities)}_dr{dropout:.2f}"

# Samle nøkkelinnstillinger
settings = {
    "Timestamp": timestamp,
    "Batch Size": bs,
    "Image Size": img_size,
    "Modalities": ", ".join(params.selected_modalities),
    "Train Samples": len(train_loader.dataset),
    "Validation Samples": len(val_loader.dataset),
    "Test Samples": len(dftest),
    "Model Architecture": params.modelname,
    "Spatial Dimensions": 3,
    "Dropout": dropout,
    "Loss Function": "DiceLossWithSigmoid",
    "Metric": "ThresholdedDice (monitor='dice')",
    "Callback": "EarlyStoppingCallback(monitor='dice', comp=np.greater, patience=10)",
    "Optimizer": "fit_one_cycle",
    "Epochs": n_epoch,
    "Learning Rate": lr,
    "Model Path": model_path
}

# Lagre innstillinger som CSV
settings_df = pd.DataFrame([settings])
pathsave = os.path.join(params.prepathmodels, f"{basename}.csv")
settings_df.to_csv(pathsave, index=False)
print(f"✅ Lagret treningsinnstillinger til: {pathsave}")

# Lagre også trenings- og testdataframes
path_train = os.path.join(params.prepathmodels, f"{basename}_dftrain.csv")
path_test  = os.path.join(params.prepathmodels, f"{basename}_dftest.csv")

dftrain.to_csv(path_train, index=False)
dftest.to_csv(path_test, index=False)

print(f"📂 Lagret dftrain: {path_train}")
print(f"📂 Lagret dftest:  {path_test}")

# Vis sammendrag i notebook
display(settings_df)


In [ ]:
# Load the model
# Define the model architecture (same as used for training)
model = monai_unet_model(dropout=0.2)

# Load the saved weights
pathload = os.path.join(prepathmodels, f'UNet-Monai-20241119-{version}.pth')
print(f'Loading model weights from {pathload}')
model.load_state_dict(torch.load(pathload))
model.eval()  # Set the model to evaluation mode


In [ ]:
learn.recorder.plot_loss()

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt

def compute_dice_score(preds, targets, threshold=0.5, apply_sigmoid=True):
    if apply_sigmoid:
        preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    targets = targets.float()
    eps = 1e-8
    dice_scores = []
    for i in range(preds.shape[0]):
        inter = (preds[i] * targets[i]).sum().float()
        union = preds[i].sum().float() + targets[i].sum().float()
        dice = (2 * inter + eps) / (union + eps)
        dice_scores.append(dice.item())
    return dice_scores


def show_predictions_with_total_uncertainty(batch, mean_pred, std_pred, dice_score, threshold=0.5):
    """
    Viser for hvert volum:
      1. originalbilde
      2. ground truth
      3. sannsynlighetskart (mean_pred)
      4. tersklet binærmaske
      5. samlet usikkerhet (kombinert)
    """
    images, gts = batch
    images, gts, mean_pred, std_pred = [x.cpu().numpy() for x in (images, gts, mean_pred, std_pred)]
    num_images = images.shape[0]
    fig, axes = plt.subplots(num_images, 5, figsize=(24, 5*num_images))
    axes = np.array(axes).reshape(num_images, 5)

    for i in range(num_images):
        img = images[i, 0]
        gt  = gts[i, 0]
        prob = mean_pred[i, 0]            # gjennomsnittlig sannsynlighet (0–1)
        mask = (prob > threshold).astype(float)
        # samlet usikkerhet = p≈0.5 + variasjon mellom MC-run
        total_uncertainty = 1 - np.abs(2*prob - 1) + std_pred[i, 0]
        total_uncertainty = np.clip(total_uncertainty, 0, 1)

        # finn slice med mest maske
        z = gt.sum(axis=(0, 1)).argmax()

        axes[i,0].imshow(img[:,:,z], cmap='gray'); axes[i,0].set_title(f'Image {i}'); axes[i,0].axis('off')
        axes[i,1].imshow(gt[:,:,z], cmap='gray'); axes[i,1].set_title('Ground Truth'); axes[i,1].axis('off')
        axes[i,2].imshow(prob[:,:,z], cmap='gray', vmin=0, vmax=1)
        axes[i,2].set_title('Predicted Prob (0–1)'); axes[i,2].axis('off')
        axes[i,3].imshow(mask[:,:,z], cmap='gray', vmin=0, vmax=1)
        axes[i,3].set_title(f'Thresholded (> {threshold:.2f})\nDice={dice_score[i]:.2f}'); axes[i,3].axis('off')
        im = axes[i,4].imshow(total_uncertainty[:,:,z], cmap='hot', vmin=0, vmax=1)
        axes[i,4].set_title('Total Uncertainty'); axes[i,4].axis('off')

    plt.tight_layout()
    plt.show()


# ==========================================================
# Eval-loop med Monte Carlo dropout
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learn.model.to(device)
learn.model.train()   # aktiver dropout under inferens

num_mc = 10
dice_scores = []

with torch.no_grad():
    for batch in val_loader:
        images, targets = [b.to(device) for b in batch]

        # -- Monte Carlo Dropout
        preds_list = []
        for _ in range(num_mc):
            preds = torch.sigmoid(learn.model(images))
            preds_list.append(preds.unsqueeze(0))
        preds_stack = torch.cat(preds_list, dim=0)

        mean_pred = preds_stack.mean(dim=0)
        std_pred = preds_stack.std(dim=0)

        batch_dice = compute_dice_score(mean_pred, targets, apply_sigmoid=False)
        dice_scores.extend(batch_dice)

        # vis med samlet usikkerhet
        show_predictions_with_total_uncertainty((images, targets), mean_pred, std_pred, batch_dice, threshold=0.5)

print(f"Average Dice Score: {np.mean(dice_scores):.4f}")


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, nibabel as nib

def compute_dice_score(preds, targets, threshold=0.5, apply_sigmoid=True):
    if apply_sigmoid:
        preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()
    targets = targets.float()
    eps = 1e-8
    dice_scores = []
    for i in range(preds.shape[0]):
        intersection = (preds[i] * targets[i]).sum().float()
        union = preds[i].sum().float() + targets[i].sum().float()
        dice = (2 * intersection + eps) / (union + eps)
        dice_scores.append(dice.item())
    return dice_scores


def show_max_mask_predictions(batch, predictions, dice_score, threshold=0.5):
    """
    Viser for hvert volum:
      1. originalbilde
      2. ground truth
      3. sannsynlighetskart (etter sigmoid)
      4. tersklet binærmaske (> threshold)
    """
    images, gts = batch
    images, gts, predictions = [x.cpu().numpy() for x in (images, gts, predictions)]
    num_images = images.shape[0]
    fig, axes = plt.subplots(num_images, 4, figsize=(20, 5*num_images))
    axes = np.array(axes).reshape(num_images, 4)

    for i in range(num_images):
        img = images[i, 0]
        gt  = gts[i, 0]
        pred_prob = predictions[i, 0]              # sigmoid output (0–1)
        pred_mask = (pred_prob > threshold).astype(float)

        # Finn snitt med mest maske
        slice_sums = gt.sum(axis=(0, 1))
        z = slice_sums.argmax()

        # 1️⃣ Originalbilde
        axes[i,0].imshow(img[:,:,z], cmap='gray')
        axes[i,0].set_title(f'Image {i}')
        axes[i,0].axis('off')

        # 2️⃣ Ground Truth
        axes[i,1].imshow(gt[:,:,z], cmap='gray')
        axes[i,1].set_title('Ground Truth')
        axes[i,1].axis('off')

        # 3️⃣ Sannsynlighetskart (0–1)
        axes[i,2].imshow(pred_prob[:,:,z], cmap='gray', vmin=0, vmax=1)
        axes[i,2].set_title('Predicted Prob (0–1)')
        axes[i,2].axis('off')

        # 4️⃣ Tersklet binærmaske
        axes[i,3].imshow(pred_mask[:,:,z], cmap='gray', vmin=0, vmax=1)
        axes[i,3].set_title(f'Thresholded (> {threshold:.2f})\nDice={dice_score[i]:.2f}')
        axes[i,3].axis('off')

    plt.tight_layout()
    plt.show()


# ==========================================================
# Eval-loop (uendret, men nå 4 kolonner i plot)
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learn.model.to(device).eval()

dice_scores = []
with torch.no_grad():
    for batch in val_loader:
        images, targets = [b.to(device) for b in batch]

        # 👉 Gjør sigmoid her, én gang for alle
        preds = torch.sigmoid(learn.model(images))

        # Beregn Dice uten å sigmoid'e på nytt
        batch_dice = compute_dice_score(preds, targets, apply_sigmoid=False)

        # Plot med ekstra kolonne for binærmaske
        show_max_mask_predictions((images, targets), preds, batch_dice, threshold=0.5)

        dice_scores.extend(batch_dice)

print(f"Average Dice Score: {np.mean(dice_scores):.4f}")


In [ ]:
dice_scores